In [49]:
#Import the libraries 
import os 
import spacepy.pycdf as pycdf
import numpy as np 
import pandas as pd 

In [ ]:
#Path to saved PAD data 
path = '/Users/quocanh1306/Downloads/3.0 (1)' 

#List of years and months data 
years = ['2015', '2016']
months = ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']
#years = [1] #mock list for test run 
#months = [1] #mock list for test run 

In [ ]:
#The empty list of necessary columns 
b_azim = [] 
b_elev = []
diff_en_fluxes = []
energy = []
epoch = []
pa = []

In [ ]:
#Run this loop to read the files and necessary columns 
for year in years: 
 path = path + f'/{year}'
 for month in months: 
    path = path + f'/{month}'
    for filename in os.listdir(path): 
      if filename.endswith('.cdf'):
       print(filename)
       filepath = os.path.join(path, filename)
       readfile = pycdf.CDF(filepath) 
       b_azim.append(readfile['b_azim'][:])
       b_elev.append(readfile['b_azim'][:])
       diff_en_fluxes.append(readfile['diff_en_fluxes'][:])
       pa.append(readfile['pa'][:])
       energy.append(readfile['energy'][:])
       epoch.append(readfile['epoch'][:])

mvn_swe_l2_svypad_20150417_v05_r01.cdf


In [ ]:
#Optional: Run this to check the data in the columns 
print(b_azim) #shape = (30870,1)
print(b_elev) #shape = (30870,1)
print(diff_en_fluxes) #shape = (30870, 16, 64)
print(energy) #shape = (64,1)
print(epoch) #shape = (30870,1)
print(pa) #shape = (30870, 16, 64)

[array([153.98436, 201.79686, 166.64061, ..., 308.67184, 311.48438,
       311.48438], shape=(30870,), dtype=float32)]
[array([153.98436, 201.79686, 166.64061, ..., 308.67184, 311.48438,
       311.48438], shape=(30870,), dtype=float32)]
[[[0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 7.33979440e+07
   1.42680784e+08 1.42680784e+08]
  [0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 7.30496960e+07
   1.26747048e+08 1.26747048e+08]
  [0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 1.33558776e+08
   2.36079344e+08 2.36079344e+08]
  ...
  [0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 2.39728560e+08
   2.26277904e+08 2.26277904e+08]
  [0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 1.26461888e+08
   1.24311760e+08 1.24311760e+08]
  [0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 1.13424256e+08
   1.70821424e+08 1.70821424e+08]]

 [[0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 9.23879840e+07
   7.72703520e+07 7.72703520e+07]
  [0.00000000e+00 0.00000000e+00 0.00000000e+00

In [ ]:
#Convert other columns into numpy array type 
b_azim = np.squeeze(np.array(b_azim))
b_elev = np.squeeze(np.array(b_elev))
energy = np.squeeze(np.array(energy))
epoch = np.squeeze(np.array(epoch))
diff_en_fluxes = np.squeeze(np.array(diff_en_fluxes))
pa = np.squeeze(np.array(pa))

In [ ]:
#Create a multi-index dataframe 
names = ['epoch', 'pa bin', 'energy']
index = pd.MultiIndex.from_product([range(s)for s in diff_en_fluxes.shape], names = names)

# Replace the integer levels with actual values
index = index.set_levels([
    epoch,                   
    index.levels[1],                # Keep pa bin level as is
    energy                  
])

df = pd.DataFrame({'def': diff_en_fluxes.flatten(), 
                   'pa values': pa.flatten(),
                   'b_azim': np.repeat(b_azim, 16 * 64), 
                   'b_elev': np.repeat(b_azim, 16 * 64)},
                   index=index)

# Reset the indices 
df = df.reset_index()
df = df.set_index(['epoch', 'b_azim', 'b_elev', 'pa bin', 'energy'])

#Save to CSV
df.to_csv('PAD_csv.csv', index = False)

#Save to HDF 
df.to_hdf('PAD_hdf.h5', key='data', mode='w')

/Users/quocanh1306/Desktop/miniconda3/envs/py310/lib/python3.10/site-packages/tables/attributeset.py:462: NaturalNameWarning: object name is not a valid Python identifier: 'axis1_namepa bin'; it does not match the pattern ``^[a-zA-Z_][a-zA-Z0-9_]*$``; you will not be able to use natural naming to access this object; using ``getattr()`` will still work, though
  check_attribute_name(name)
/var/folders/fw/_vv3v5xx4237m2_lkjg3_0s00000gn/T/ipykernel_33799/2747335416.py:23: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block1_values] [items->Index(['pa values'], dtype='object')]

  df.to_hdf('PAD_hdf.h5', key='data', mode='w')
